In [23]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import torch

In [24]:
path = "./files/df_ml_good_with_features.csv"
df = pd.read_csv(path)
df_amyloid = df[df["class"] == 1]

print(df_amyloid.columns)

Index(['id', 'sequence', 'length', 'class', 'merge_id', 'a3vSA', 'AAT',
       'Na4vSS', 'THSA', 'nHS', 'TA', 'net_charge', 'hydrophobicity',
       'polar_fraction', 'nonpolar_fraction', 'acidic_fraction',
       'basic_fraction', 'aromatic_fraction', 'beta_propensity', 'seq_entropy',
       'proline_fraction'],
      dtype='object')


In [25]:
features_map = {
    'beta_propensity': 'bp',
    'proline_fraction': 'pf',
    'AAT': 'aat',
    'net_charge': 'nc',
    'TA': 'ta',
    'polar_fraction': 'pol',
    'a3vSA': 'sa'
}

def build_input(example):
    feat_str = "|".join([
        f"{features_map[f]}={example[f]:.2f}"
        for f in features_map
    ])
    return feat_str + " <SEP> " + example["sequence"]

df_amyloid["text"] = df_amyloid.apply(build_input, axis=1)

C:\Users\marts\AppData\Local\Temp\ipykernel_30460\1106537780.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_amyloid["text"] = df_amyloid.apply(build_input, axis=1)


In [26]:
dataset = Dataset.from_pandas(df_amyloid[['text']])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [27]:
print((train_dataset[0]["text"]))

bp=0.92|pf=0.00|aat=1.43|nc=-1.90|ta=-5.18|pol=0.18|sa=-0.26 <SEP> QARDSGEIYAAAGDMHI


In [28]:
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
tokenizer.pad_token = tokenizer.eos_token

lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df_amyloid["text"].sample(1000)
]

print(max(lengths), sum(lengths)/len(lengths))

81 69.487


In [29]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],   # lista stringów
        truncation=True,
        padding="max_length",
        max_length=90
    )

    labels = []
    for seq in tokens["input_ids"]:
        labels.append([
            -100 if token == tokenizer.pad_token_id else token
            for token in seq
        ])

    tokens["labels"] = labels
    return tokens


train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/905 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

In [30]:
for i in range(5):
    print(train_dataset[i])

{'text': 'bp=0.92|pf=0.00|aat=1.43|nc=-1.90|ta=-5.18|pol=0.18|sa=-0.26 <SEP> QARDSGEIYAAAGDMHI', '__index_level_0__': 380, 'input_ids': [66, 80, 29, 16, 14, 25, 18, 92, 80, 70, 29, 16, 14, 16, 16, 92, 65, 65, 84, 29, 17, 14, 20, 19, 92, 78, 67, 29, 13, 17, 14, 25, 16, 92, 84, 65, 29, 13, 21, 14, 17, 24, 92, 80, 79, 76, 29, 16, 14, 17, 24, 92, 83, 65, 29, 13, 16, 14, 18, 22, 221, 28, 2856, 30, 221, 1156, 36, 784, 399, 1888, 594, 553, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [66, 80, 29, 16, 14, 25, 18, 92, 80, 70, 29, 16, 14, 16, 16, 92, 65, 65, 84, 29, 17, 14, 20, 19, 92, 78, 67, 29, 13, 17, 14, 25, 16, 92, 84, 65, 29, 13, 21, 14, 17, 24, 92, 80, 79, 76, 29, 16, 14, 17, 24, 92,

In [35]:
model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2")

#zamrażanie dolnych warstw - bardzo mało danych
for param in model.transformer.h[:30].parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
training_args = TrainingArguments(
    output_dir="files/models/protGPT2-results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    num_train_epochs=10,
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    fp16=True, #gpu
    weight_decay=0.01, #L2
)

model.config.resid_pdrop = 0.2
model.config.embd_pdrop = 0.2
model.config.attn_pdrop = 0.2

In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [23]:
model_path = "files/models/protGPT2-results/checkpoint-906"

tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
model = AutoModelForCausalLM.from_pretrained(model_path)

model.eval()

prompt = "G"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=50,
        min_length=6,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        num_return_sequences=10
    )

sequences = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

for i, seq in enumerate(sequences):
    print(f"\nSequence {i+1}:")
    print(seq)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



Sequence 1:
GHIYQNNSVSGGTQHAPVVQGHTF

Sequence 2:
GNIVIGDNA


Sequence 3:
GVAVRGDGAVMVGNV

Sequence 4:
GNITFTGDA


Sequence 5:
GYQTGPVVQGRDFSG

Sequence 6:
GNIAAGNLTVGNN

Sequence 7:
GTVNLQGAVTGDVVQAGRDISG

Sequence 8:
GQIYQGSI
KGRARVAQG

Sequence 9:
GNVYTQNYTFQ

Sequence 10:
GNVVTAXQ
